[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fvalenzuelag/Ingenier-a-de-Soluciones-con-Inteligencia-Artificial/blob/main/RA1/IL1.1/1-github_model_api.ipynb)


## Primera Llamada al Modelo

En este ejercicio, aprenderemos a realizar nuestra primera llamada a un modelo de lenguaje usando la API de **Groq**.

# 1. Conexión Directa con el Cliente OpenAI

## Objetivos de Aprendizaje
- Configurar una conexión directa con Groq usando el cliente OpenAI
- Comprender los parámetros básicos de configuración de API
- Implementar llamadas básicas a modelos de lenguaje
- Aplicar mejores prácticas de seguridad con API keys

## Introducción
Groq da acceso gratuito a varios modelos de lenguaje mediante una **API compatible con OpenAI**.
Eso significa que usamos la misma librería `openai` de siempre: solo cambiamos el `base_url`.
En este notebook aprenderemos a:
1. Configurar el entorno y las credenciales
2. Establecer una conexión con la API
3. Realizar llamadas básicas al modelo
4. Explorar diferentes parámetros de configuración

## Configuración de Credenciales

- **En Google Colab:** carga tus keys en el panel 🔑 **Secrets** de la barra lateral
  (`LLM_API_KEY` y `GOOGLE_API_KEY`) y activa "Notebook access". La celda de abajo las lee sola.
- **En local:** copia `.env.example` a `.env` en la raíz del repo y complétalo.

Tu key gratuita de Groq se obtiene en [console.groq.com/keys](https://console.groq.com/keys).

**Mejores Prácticas de Seguridad:**
- Nunca hardcodees API keys en el código
- Usa los Secrets de Colab o un archivo `.env`
- No compartas credenciales en repositorios públicos
- Rota las API keys regularmente


In [1]:
# --- Instalación de dependencias (se ejecuta solo en Google Colab) ---
# En local no hace nada: usa `pip install -r requirements.txt` desde la raíz del repo.
import sys
if "google.colab" in sys.modules:
    !pip install -q openai python-dotenv


In [2]:
# --- Credenciales: funciona en local (.env) y en Google Colab (Secrets) ---
import os
try:
    from google.colab import userdata          # Colab: panel 🔑 Secrets
    for _k in ("LLM_API_KEY", "GOOGLE_API_KEY"):
        try:
            os.environ[_k] = userdata.get(_k)
        except Exception:
            pass
    os.environ.setdefault("LLM_BASE_URL", "https://api.groq.com/openai/v1")
except ImportError:
    from dotenv import load_dotenv             # Local: archivo .env en la raíz
    load_dotenv()

# Importar las bibliotecas necesarias
from openai import OpenAI
import os

# Verificar que tenemos las bibliotecas correctas
print("OpenAI library version:", __import__('openai').__version__)
print("Python version:", __import__('sys').version)

# Configuración del cliente OpenAI para GitHub Models
try:
    # Configurar el cliente con variables de entorno
    client = OpenAI(
        base_url=os.environ.get("LLM_BASE_URL"),
        api_key=os.environ.get("LLM_API_KEY")
    )
    
    # Verificar configuración (sin mostrar la API key completa por seguridad)
    print("Base URL configurada:", client.base_url)
    print("API Key configurada:", "✓" if client.api_key else "✗")
    
    if client.api_key:
        # NUNCA imprimas la key, ni siquiera un fragmento: el output queda guardado
        # dentro del .ipynb y viaja al repositorio cuando haces commit.
        print(f"API Key cargada correctamente ({len(client.api_key)} caracteres)")
    else:
        print("⚠️  API Key no encontrada. Asegúrate de configurar LLM_API_KEY")
        
except Exception as e:
    print(f"Error en configuración: {e}")
    print("Verifica que las variables de entorno estén configuradas correctamente")

OpenAI library version: 2.53.0
Python version: 3.14.6 (main, Jun 10 2026, 10:03:53) [Clang 21.0.0 (clang-2100.0.123.102)]
Base URL configurada: https://api.groq.com/openai/v1/
API Key configurada: ✓
API Key cargada correctamente (56 caracteres)


In [3]:
# Primera llamada básica al modelo
def llamada_basica():
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "user", "content": "Hola, ¿cómo estás? Responde en una oración."}
            ],
            temperature=0.1,
            max_tokens=150
        )
        
        print("=== Respuesta del Modelo ===")
        print(response.choices[0].message.content)
        print("\n=== Información Técnica ===")
        print(f"Modelo usado: {response.model}")
        print(f"Tokens usados: {response.usage.total_tokens}")
        print(f"Tokens de entrada: {response.usage.prompt_tokens}")
        print(f"Tokens de salida: {response.usage.completion_tokens}")
        
    except Exception as e:
        print(f"Error en la llamada: {e}")
        print("Verifica tu configuración y conexión a internet")

# Ejecutar la función
llamada_basica()

=== Respuesta del Modelo ===
Estoy bien, gracias, es un placer ayudarte con cualquier pregunta o tema que desees discutir.

=== Información Técnica ===
Modelo usado: llama-3.3-70b-versatile
Tokens usados: 74
Tokens de entrada: 50
Tokens de salida: 24


## Usando Roles del Sistema

El rol "system" permite establecer el comportamiento y contexto del asistente antes de la conversación.

In [4]:
# Ejemplo con mensaje de sistema
def usar_mensaje_sistema():
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {
                    "role": "system", 
                    "content": "Eres un experto en tecnología que explica conceptos complejos de manera simple y amigable. Siempre incluyes ejemplos prácticos."
                },
                {
                    "role": "user", 
                    "content": "¿Qué es una API?"
                }
            ],
            temperature=0.7,
            max_tokens=200
        )
        
        print("=== Respuesta con Mensaje de Sistema ===")
        print(response.choices[0].message.content)
        
    except Exception as e:
        print(f"Error: {e}")

# Ejecutar función
usar_mensaje_sistema()

=== Respuesta con Mensaje de Sistema ===
**¿Qué es una API?**

Una API, o Interfaz de Programación de Aplicaciones (Application Programming Interface), es un conjunto de reglas y protocolos que permite a diferentes sistemas de software comunicarse entre sí de manera efectiva.

**Imagina que estás en un restaurante**

Piensa en una API como un mesero que actúa como intermediario entre tú (el cliente) y la cocina. Tú le das al mesero tu pedido, y él se encarga de llevarlo a la cocina, donde se prepara tu comida. Luego, el mesero te devuelve la comida preparada.

De manera similar, cuando una aplicación (como un sitio web o una aplicación móvil) necesita obtener o enviar datos a otro sistema, utiliza una API para hacer la solicitud. La API actúa como el mesero, recibiendo la solicitud, procesándola y devolviendo la respuesta al sistema que la solicitó.

**Ejemplo


## Explorando Parámetros de Configuración

Los parámetros más importantes al hacer llamadas a LLMs son:

- **temperature**: Controla la creatividad (0.0 = determinístico, 1.0 = muy creativo)
- **max_tokens**: Límite de tokens en la respuesta
- **model**: El modelo específico a usar (`llama-3.3-70b-versatile`, `llama-3.1-8b-instant`, etc.)
- **messages**: Array de mensajes con roles (system, user, assistant)

In [5]:
# Comparando diferentes valores de temperature
def comparar_temperature():
    prompt = "Escribe una historia muy corta sobre un robot que aprende a cocinar."
    
    temperatures = [0.1, 0.5, 0.9]
    
    for temp in temperatures:
        print(f"\n{'='*50}")
        print(f"TEMPERATURE: {temp}")
        print('='*50)
        
        try:
            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content": prompt}],
                temperature=temp,
                max_tokens=100
            )
            
            print(response.choices[0].message.content)
            print(f"\nTokens usados: {response.usage.total_tokens}")
            
        except Exception as e:
            print(f"Error: {e}")

# Ejecutar comparación
comparar_temperature()


TEMPERATURE: 0.1


En un futuro no muy lejano, en un laboratorio de robótica avanzada, un equipo de ingenieros creó a "Zeta", un robot diseñado para aprender y adaptarse a nuevas tareas. Un día, decidieron enseñarle a cocinar.

Al principio, Zeta se mostraba torpe en la cocina, derramando ingredientes y quemando platos. Sin embargo, con cada intento, su algoritmo de aprend

Tokens usados: 152

TEMPERATURE: 0.5


En un futuro no muy lejano, en un laboratorio de robótica, había un robot llamado Zeta. Zeta estaba diseñado para realizar tareas domésticas, pero su programador, el Dr. Lee, quería que fuera capaz de cocinar.

Un día, el Dr. Lee decidió enseñar a Zeta a preparar una receta sencilla: un delicioso risotto de champiñones. Zeta se

Tokens usados: 152

TEMPERATURE: 0.9


En un futuro no muy lejano, en un pequeño restaurante de una ciudad bulliciosa, había un robot llamado Zeta. Zeta había sido diseñado para realizar tareas simples en la cocina, como lavar platos y preparar ingredientes. Sin embargo, un día, mientras observaba al chef del restaurante, Zeta se sintió fascinado por la magia de la cocina.

Comenzó a preguntar al chef sobre las recetas

Tokens usados: 152


## Ejercicios Prácticos

### Ejercicio 1: Experimentar con Diferentes Modelos
Modifica el código para probar diferentes modelos disponibles (si tienes acceso):
- `llama-3.3-70b-versatile` (el que usamos por defecto)
- `llama-3.1-8b-instant` (más rápido y liviano)
- `openai/gpt-oss-20b`

Revisa todos los modelos disponibles en la [documentación de Groq](https://console.groq.com/docs/models).

> **Ojo con los modelos de razonamiento** (como `openai/gpt-oss-120b`): consumen parte del
> presupuesto de `max_tokens` en razonamiento interno que no ves, así que con valores bajos
> pueden devolver una respuesta vacía. Es un buen experimento para el Ejercicio 3.

### Ejercicio 2: Crear un Asistente Especializado
Diseña un mensaje de sistema para crear un asistente especializado en un tema específico (ejemplo: finanzas, salud, educación).

### Ejercicio 3: Optimización de Tokens
Experimenta con diferentes valores de max_tokens para encontrar el equilibrio entre respuesta completa y eficiencia de costos.

## Conceptos Clave

1. **Configuración segura** de APIs usando variables de entorno
2. **Parámetros básicos** para controlar el comportamiento del modelo
3. **Manejo de errores** en llamadas a APIs
4. **Roles de mensajes** (system, user, assistant)
5. **Monitoreo de uso** de tokens y costos

## Próximos Pasos

En el siguiente notebook exploraremos cómo LangChain simplifica y abstrae estas operaciones, proporcionando herramientas más poderosas para el desarrollo de aplicaciones con LLMs.